In [ ]:
# === Child: parameters cell ===
# This cell MUST be tagged 'parameters' (Fabric replaces these values at runtime
# with whatever the parent passes in `arguments`).
# Defaults below let the child run standalone for development/testing.

function_url = "https://my-fn-app.azurewebsites.net/api/HelloWorld"
function_key = "REPLACE_WITH_FUNCTION_KEY"
payload      = {"name": "default"}

# Child notebook: call an Azure Function

Receives parameters from a parent notebook (via the cell tagged `parameters`), calls an HTTP-triggered Azure Function, and returns the result as JSON to the parent via `notebookutils.notebook.exit()`.

In [ ]:
# === Child: call the Azure Function ===
import json, time, requests

# --- Auth options -------------------------------------------------------
# Option A (default): function key in x-functions-key header.
#   Use for HTTP-triggered functions with authLevel=function or admin.
# Option B (commented): Entra (Easy Auth) bearer token.
#   Use for functions secured via App Registration / managed identity.
#   audience = your function's App Registration application URI / client id.

headers = {
    "Content-Type":    "application/json",
    "x-functions-key": function_key,
}

# --- Option B: Entra bearer token instead of function key ---------------
# try:
#     from notebookutils.credentials import getToken
# except Exception:
#     from mssparkutils.credentials import getToken
# audience = "api://<your-function-app-client-id>"
# headers = {
#     "Content-Type":  "application/json",
#     "Authorization": f"Bearer {getToken(audience)}",
# }

t0 = time.time()
try:
    resp = requests.post(function_url, headers=headers,
                         data=json.dumps(payload), timeout=60)
    duration_ms = int((time.time() - t0) * 1000)

    # Try to parse response as JSON, else return text.
    try:
        body = resp.json()
    except ValueError:
        body = resp.text

    if resp.ok:
        result = {
            "status":      "ok",
            "http_status": resp.status_code,
            "duration_ms": duration_ms,
            "response":    body,
        }
    else:
        result = {
            "status":      "error",
            "http_status": resp.status_code,
            "duration_ms": duration_ms,
            "error":       f"HTTP {resp.status_code}: {str(body)[:500]}",
            "response":    body,
        }

except requests.RequestException as e:
    duration_ms = int((time.time() - t0) * 1000)
    result = {
        "status":      "error",
        "http_status": None,
        "duration_ms": duration_ms,
        "error":       f"{type(e).__name__}: {e}",
        "response":    None,
    }

print(json.dumps(result, indent=2))

In [ ]:
# === Child: return JSON to parent ===
import json
try:
    from notebookutils import notebook
except Exception:
    from mssparkutils import notebook

notebook.exit(json.dumps(result))